# Dự đoán khách hàng chốt sale

**Mục tiêu:** dự báo khách hàng có chốt sale hay không, huấn luyện Random Forest, kiểm tra chỉ số và biến quan trọng.

- Nhãn: `1` nếu `thoi_gian_chuyen_sang_chot_sale` có giá trị; ngược lại là `0`.
- Chia có phân tầng: **80% train / 15% test / 5% holdout**.
  - **Train (80%)**: mô hình học.
  - **Test (15%)**: kiểm tra chất lượng trong lúc làm việc.
  - **Holdout (5%)**: giữ nguyên từ dữ liệu gốc, mô hình chưa từng thấy; chỉ dùng ở bước cuối để so sánh dự báo với thực tế.
- Mô hình giả định các biến tư vấn/chăm sóc đã có trước khi chốt. Nếu dự đoán ngay lúc đăng ký, cần bỏ các biến ghi nhận sau đăng ký.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
data = pd.read_excel("Courses_of_Interest_Processed.xlsx")
target_col = "thoi_gian_chuyen_sang_chot_sale"
y = data[target_col].notna().astype(int)
print("Data shape:", data.shape)
print("Target counts:", y.value_counts().to_dict())

## Đặc trưng thời gian

Tách giờ, thứ và tháng từ ngày đăng ký; thêm trạng thái và khoảng ngày đến tư vấn. Không đưa thời điểm chốt hoặc cột phát sinh từ thời điểm chốt vào mô hình.

In [ ]:
signup_time = pd.to_datetime(data["ngay_tao"], errors="coerce")
consult_time = pd.to_datetime(data["thoi_gian_chuyen_sang_tu_van"], errors="coerce")
data["signup_hour"] = signup_time.dt.hour
data["signup_weekday"] = signup_time.dt.dayofweek
data["signup_month"] = signup_time.dt.month

Các đặc trưng mới chỉ dùng ngày đăng ký và ngày chuyển tư vấn. Giá trị ngày âm được đánh dấu thiếu, không đổi thành trị tuyệt đối.

In [ ]:
data["signup_weekend"] = data["signup_weekday"].isin([5, 6]).astype(int)
data["has_consult"] = consult_time.notna().astype(int)
data["days_to_consult"] = (consult_time - signup_time).dt.total_seconds() / 86400
data.loc[data["days_to_consult"] < 0, "days_to_consult"] = np.nan
print("Missing days-to-consult:", int(data["days_to_consult"].isna().sum()))

## Biến đầu vào

Dùng loại chăm sóc, nhà mạng, khóa học gốc, sentiment, tư vấn viên, cờ tên thiếu, đặc trưng tư vấn và lịch đăng ký. Bỏ ID/STT, thời điểm chốt, mã khóa suy ra, cờ lỗi hằng, `tao_lich_tu_van`/`check_zalo` gần hằng và `hieu_suat` thiếu nhiều.

In [ ]:
categorical_cols = ["MKT_xu_ly_data", "nha_mang", "quan_tam_khoa_hoc", "sentimatic", "tu_van_vien"]
numeric_cols = ["loi_ten_rong", "has_consult", "days_to_consult", "signup_hour", "signup_weekday", "signup_month", "signup_weekend"]
feature_cols = categorical_cols + numeric_cols
X = data[feature_cols].copy()
X[categorical_cols] = X[categorical_cols].fillna("Chua ro").astype(str)

## Chia dữ liệu 80 / 15 / 5

Tách trước 5% làm **holdout** (giữ nguyên dữ liệu gốc, chưa cho mô hình học). Phần 95% còn lại chia thành 80% train và 15% test (15/95 của phần còn lại). `stratify` giữ tỷ lệ chốt gần nhau giữa 3 tập.

In [ ]:
X_dev, X_hold, y_dev, y_hold = train_test_split(X, y, test_size=0.05, stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_dev, y_dev, test_size=15/95, stratify=y_dev, random_state=42)

split_summary = pd.DataFrame({
    "Số dòng": [len(X_train), len(X_test), len(X_hold)],
    "Tỷ lệ": [len(X_train)/len(X), len(X_test)/len(X), len(X_hold)/len(X)],
    "Số khách chốt": [y_train.sum(), y_test.sum(), y_hold.sum()],
    "Tỷ lệ chốt": [y_train.mean(), y_test.mean(), y_hold.mean()],
}, index=["Train (80%)", "Test (15%)", "Holdout (5%)"])
display(split_summary.round(3))

## Ma trận tương quan

Tính Pearson trên biến số/binary của train và nhãn. Biến danh mục không đưa vào Pearson vì mã danh mục không có thứ tự số học; chúng sẽ được one-hot encode cho Random Forest.

In [ ]:
corr_data = X_train[numeric_cols].copy()
corr_data = corr_data.fillna(corr_data.median())
corr_data["chot"] = y_train
corr_matrix = corr_data.corr()
display(corr_matrix.round(2))

`signup_weekday` và `signup_weekend` có liên hệ vì weekend được tạo từ thứ trong tuần. Ma trận chỉ phản ánh tương quan tuyến tính; mức quan trọng cuối cùng sẽ xem từ Random Forest.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Train: numeric and binary feature correlations")
plt.tight_layout()
plt.show()

## Preprocessing

Median cho cột số, giá trị xuất hiện nhiều nhất cho danh mục, sau đó one-hot encode. Bộ xử lý chỉ `fit` trên train; test và holdout chỉ `transform`.

In [ ]:
numeric_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
])

In [ ]:
category_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=25, drop="first", sparse_output=False)),
])

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_cols),
    ("category", category_pipe, categorical_cols),
])

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
print("Encoded train / test:", X_train_encoded.shape, X_test_encoded.shape)

## Huấn luyện Random Forest

Train mô hình trên 80% dữ liệu. Tập test 15% dùng để xem chất lượng mô hình; holdout 5% vẫn để riêng đến bước cuối.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300, max_features="sqrt",
    min_samples_leaf=5, class_weight="balanced",
    random_state=42, n_jobs=-1,
)
rf_model.fit(X_train_encoded, y_train)
print("Đã huấn luyện xong trên", len(X_train), "dòng")

## Kiểm tra trên train và test (15%)

Dùng ngưỡng phân loại mặc định 0,5. ROC-AUC và PR-AUC dùng xác suất. So sánh train với test để biết mô hình có bị học thuộc (overfit) không. Holdout vẫn chưa được xem.

In [ ]:
def evaluate(model, X_enc, y_true, threshold=0.5):
    proba = model.predict_proba(X_enc)[:, 1]
    pred = (proba >= threshold).astype(int)
    metrics = pd.Series({
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
    })
    return metrics, proba, pred

train_metrics, train_probability, train_prediction = evaluate(rf_model, X_train_encoded, y_train)
test_metrics, test_probability, test_prediction = evaluate(rf_model, X_test_encoded, y_test)
display(pd.DataFrame({"Train (80%)": train_metrics, "Test (15%)": test_metrics}).round(3))

### Confusion matrix test

Hàng là nhãn thật, cột là dự đoán. Dùng để xem mô hình bỏ sót khách chốt hay báo nhầm khách chưa chốt.

In [ ]:
test_matrix = confusion_matrix(y_test, test_prediction)
ConfusionMatrixDisplay(test_matrix, display_labels=["No sale", "Sale"]).plot(cmap="Blues", values_format="d")
plt.title("Test (15%)")
plt.tight_layout()
plt.show()

## Biến quan trọng

Random Forest trả importance cho từng cột sau one-hot. Gộp importance theo tên biến gốc để dễ xem biến nào đóng góp nhiều nhất.

In [ ]:
def source_feature(encoded_name):
    for column in categorical_cols:
        if encoded_name.startswith(f"category__{column}_"):
            return column
    return encoded_name.replace("numeric__", "")

In [ ]:
importance = pd.DataFrame({"feature": feature_names, "importance": rf_model.feature_importances_})
importance["variable"] = importance["feature"].apply(source_feature)
importance_by_variable = importance.groupby("variable", as_index=False)["importance"].sum()
importance_by_variable = importance_by_variable.sort_values("importance", ascending=False)
display(importance_by_variable.head(12).round(4))

In [ ]:
top_importance = importance_by_variable.head(10).sort_values("importance")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_importance["variable"], top_importance["importance"])
ax.set_title("Random Forest: variable importance")
plt.tight_layout()
plt.show()

## Dự báo trên 5% holdout (dữ liệu mô hình chưa từng học)

Tập này không dùng để chọn biến, chỉnh tham số hay chọn ngưỡng. Mô hình dự báo từng khách, rồi đối chiếu với kết quả thực tế (có chốt hay không) trong dữ liệu gốc.

In [ ]:
X_hold_encoded = preprocessor.transform(X_hold)
hold_metrics, hold_probability, hold_prediction = evaluate(rf_model, X_hold_encoded, y_hold)
print("Holdout rows:", len(hold_prediction))
display(hold_metrics.round(3))

### So sánh chỉ số: Train vs Test vs Holdout

In [ ]:
compare = pd.DataFrame({
    "Train (80%)": train_metrics,
    "Test (15%)": test_metrics,
    "Holdout (5%)": hold_metrics,
})
compare["Chênh lệch Holdout - Test"] = compare["Holdout (5%)"] - compare["Test (15%)"]
display(compare.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (title, yt, yp) in zip(axes, [("Test (15%)", y_test, test_prediction), ("Holdout (5%)", y_hold, hold_prediction)]):
    ConfusionMatrixDisplay(confusion_matrix(yt, yp), display_labels=["No sale", "Sale"]).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### Đối chiếu thực tế và dự báo từng khách trong holdout

Bảng dưới cho thấy từng khách: thực tế chốt hay không, mô hình dự báo gì và xác suất chốt. Cột `ket_qua` cho biết mô hình đúng hay sai.

In [ ]:
holdout_result = data.loc[X_hold.index, ["STT", "khach_hang", "tu_van_vien", "quan_tam_khoa_hoc", "MKT_xu_ly_data", "ngay_tao"]].copy()
holdout_result["thuc_te"] = y_hold.map({1: "Chốt", 0: "Không chốt"})
holdout_result["du_bao"] = pd.Series(hold_prediction, index=X_hold.index).map({1: "Chốt", 0: "Không chốt"})
holdout_result["xac_suat_chot"] = hold_probability.round(3)
holdout_result["ket_qua"] = np.where(holdout_result["thuc_te"] == holdout_result["du_bao"], "Đúng", "Sai")
holdout_result = holdout_result.sort_values("xac_suat_chot", ascending=False)
display(holdout_result.head(15))
print("Số dự báo đúng / tổng:", (holdout_result["ket_qua"] == "Đúng").sum(), "/", len(holdout_result))

In [ ]:
cross = pd.crosstab(holdout_result["thuc_te"], holdout_result["du_bao"], margins=True, margins_name="Tổng")
display(cross)

### Độ tin cậy của kết quả holdout

Holdout chỉ ~5% (khoảng 949 dòng) nên chỉ số có dao động. Bootstrap 1.000 lần cho khoảng tin cậy 95%; nếu khoảng này chứa chỉ số của test thì hai kết quả nhất quán.

In [ ]:
rng = np.random.default_rng(42)
y_h = y_hold.to_numpy()
acc_boot, auc_boot = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_h), len(y_h))
    acc_boot.append(accuracy_score(y_h[idx], hold_prediction[idx]))
    auc_boot.append(roc_auc_score(y_h[idx], hold_probability[idx]))
ci = pd.DataFrame({
    "Holdout": [hold_metrics["Accuracy"], hold_metrics["ROC-AUC"]],
    "CI 95% thấp": [np.percentile(acc_boot, 2.5), np.percentile(auc_boot, 2.5)],
    "CI 95% cao": [np.percentile(acc_boot, 97.5), np.percentile(auc_boot, 97.5)],
    "Test (15%)": [test_metrics["Accuracy"], test_metrics["ROC-AUC"]],
}, index=["Accuracy", "ROC-AUC"])
display(ci.round(3))

In [ ]:
holdout_result.to_excel("Holdout_5pct_DuBao_vs_ThucTe.xlsx", index=False)
compare.round(4).to_excel("So_Sanh_Chi_So_Train_Test_Holdout.xlsx")
print("Đã xuất: Holdout_5pct_DuBao_vs_ThucTe.xlsx và So_Sanh_Chi_So_Train_Test_Holdout.xlsx")

## Kết luận và lưu ý

- Nếu chỉ số holdout gần với test (và nằm trong khoảng tin cậy), mô hình tổng quát hóa tốt trên dữ liệu chưa từng thấy.
- Nếu train cao hơn nhiều so với test/holdout, mô hình đang học thuộc dữ liệu.
- `tu_van_vien` và `signup_month` chiếm importance lớn nhất: mô hình đang học theo tư vấn viên và mùa vụ/thời điểm đăng ký. Chia ngẫu nhiên có thể làm kết quả đẹp hơn thực tế nếu tư vấn viên hoặc giai đoạn cũ không còn giống hiện tại; muốn kiểm tra chắc hơn, thử chia theo thời gian (train tháng cũ, holdout tháng mới nhất).
- Ngưỡng 0,5 và `class_weight="balanced"` ưu tiên Recall (ít bỏ sót khách chốt) nên Precision thấp hơn; có thể chỉnh ngưỡng trên tập test (không dùng holdout) tùy mục tiêu kinh doanh.